# Game of Life transition-law MSPD optimization

This notebook is the entry point for the full experiment: configure it, run the GA, save artifacts, inspect plots, and watch the complex/typical Life videos from here.

The experiment optimizes Conway Life initial states using an MSPD analog over local empirical transition laws. The local object is `A_t(i) = (3x3 neighborhood at t, x_{t+1}(i))`, encoded as a 10-bit categorical symbol. No velocity or displacement metric is used.

In [1]:
from dataclasses import replace
from datetime import datetime
from pathlib import Path
import importlib
import sys

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents]
REPO_ROOT = next(
    path for path in repo_candidates
    if (path / "scripts" / "gol_transition_mspd_experiment.py").exists()
)
sys.path.insert(0, str(REPO_ROOT / "scripts"))

import gol_transition_mspd_experiment as gol_mspd
gol_mspd = importlib.reload(gol_mspd)
from gol_transition_mspd_experiment import (
    ExperimentConfig,
    lifelike_rule_label,
    run_experiment,
    run_rule_sweep_experiment,
    simulate_life,
)

REPO_ROOT

PosixPath('/Users/enrifermi/Projects/asal-fork')

In [2]:
# Change these switches for normal use, then run the notebook top to bottom.
# "full" uses the requested defaults. "quick" is a short local sanity run.
# "large" is the longer run for the next pass.
RUN_PRESET = "large"  # "quick", "full", or "large"
VERBOSE = True
PROGRESS_EVERY = 1
BACKEND = "jax"  # "jax" or "numpy"
EVAL_BATCH_SIZE = 32
JAX_METRIC_BATCH_SIZE = 0  # 0 = vmap+jit metric over the full rollout batch for max speed.
PAIR_SAMPLE = 512  # 0 means exact all-pairs; 512-1024 is the practical large-run range.
MIN_DELTAH_NONZERO_FRAC = 0.5  # keep only simulations with DeltaH > 0 for >50% windows.
DELTAH_NONZERO_EPS = 0.0
# The first JAX batch includes compilation time; later batches reuse the compiled rollout.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

FULL_CONFIG = ExperimentConfig(
    L=64,
    T=512,
    burn_in=64,
    window_size=32,
    window_step=8,
    n_cell_sample=256,
    null_reps=4,
    population_size=32,
    generations=50,
    elite_frac=0.1,
    mutation_rate=0.01,
    initial_density=0.25,
    random_seed=0,
    backend=BACKEND,
    eval_batch_size=EVAL_BATCH_SIZE,
    jax_metric_batch_size=JAX_METRIC_BATCH_SIZE,
    pair_sample=PAIR_SAMPLE,
    min_delta_h_nonzero_frac=MIN_DELTAH_NONZERO_FRAC,
    delta_h_nonzero_eps=DELTAH_NONZERO_EPS,
    verbose=VERBOSE,
    progress_every=PROGRESS_EVERY,
    output_dir=str(REPO_ROOT / "analysis" / "results" / "gol_transition_mspd" / f"notebook_full_{RUN_ID}"),
)

QUICK_CONFIG = replace(
    FULL_CONFIG,
    L=24,
    T=64,
    burn_in=8,
    window_size=8,
    window_step=4,
    n_cell_sample=32,
    null_reps=1,
    population_size=6,
    generations=3,
    random_controls=6,
    video_top_k=1,
    output_dir=str(REPO_ROOT / "analysis" / "results" / "gol_transition_mspd" / f"notebook_quick_{RUN_ID}"),
)

LARGE_CONFIG = replace(
    FULL_CONFIG,
    T=2048,
    burn_in=128,
    window_size=64,
    window_step=16,
    n_cell_sample=256,
    null_reps=2,
    population_size=64,
    generations=80,
    eval_batch_size=16,
    pair_sample=1024,
    random_controls=64,
    progress_every=1,
    output_dir=str(REPO_ROOT / "analysis" / "results" / "gol_transition_mspd" / f"notebook_large_{RUN_ID}"),
)

CONFIGS = {"quick": QUICK_CONFIG, "full": FULL_CONFIG, "large": LARGE_CONFIG}
CONFIG = CONFIGS[RUN_PRESET]
CONFIG

ExperimentConfig(L=64, T=2048, burn_in=128, window_size=64, window_step=16, n_cell_sample=256, null_reps=2, population_size=64, generations=80, elite_frac=0.1, mutation_rate=0.01, initial_density=0.25, random_seed=0, backend='jax', eval_batch_size=16, jax_metric_batch_size=0, distance='js', pooled_null=True, pair_sample=1024, delta_h_floor=0.0, min_delta_h_nonzero_frac=0.5, delta_h_nonzero_eps=0.0, mspd_floor=1e-06, eps=1e-12, tournament_size=3, random_controls=64, output_dir='/Users/enrifermi/Projects/asal-fork/analysis/results/gol_transition_mspd/notebook_large_20260522_031703', save_videos=True, video_top_k=3, video_fps=12, video_scale=8, video_stride=1, montage_frames=16, verbose=True, progress_every=1)

In [ ]:
# This cell runs the whole experiment: GA, random controls, NPZ/CSV/PNG outputs, and videos.
result = run_experiment(CONFIG)
result["summary"]

Starting GoL transition-law MSPD experiment: L=64, T=512, burn_in=64, windows=32/8, population=32, generations=50, backend=jax, eval_batch_size=8, distance=js, pooled_null=True, pair_sample=512, output_dir=/Users/enrifermi/Projects/asal-fork/analysis/results/gol_transition_mspd/notebook_full_20260522_012652
Evaluating generation 1/50...
  new global best: generation=0 candidate=0 MSPD=0.005882
  new global best: generation=0 candidate=1 MSPD=0.019789
  new global best: generation=0 candidate=29 MSPD=0.021656
generation 1/50: best=0.021656 mean=0.010413 median=0.008934 global_best=0.021656
Evaluating generation 2/50...
  new global best: generation=1 candidate=1 MSPD=0.021811
  new global best: generation=1 candidate=24 MSPD=0.026488
generation 2/50: best=0.026488 mean=0.011861 median=0.010801 global_best=0.026488
Evaluating generation 3/50...
  new global best: generation=2 candidate=0 MSPD=0.028605
  new global best: generation=2 candidate=10 MSPD=0.032331
generation 3/50: best=0.0323

In [ ]:
from IPython.display import Image, display
import pandas as pd

out_dir = Path(result["summary"]["output_dir"])
print(out_dir)

display(pd.read_csv(out_dir / "best_per_generation.csv").tail())
display(pd.read_csv(out_dir / "generation_scores.csv").tail())
display(pd.read_csv(out_dir / "random_control_scores.csv").describe())

plot_names = [
    "best_fitness_over_generations.png",
    "best_DeltaH_trace.png",
    "best_mspd_by_scale.png",
    "best_life_montage.png",
    "optimized_vs_random_score_boxplot.png",
]
for name in plot_names:
    display(Image(filename=str(out_dir / "plots" / name)))

In [ ]:
import numpy as np

best = np.load(out_dir / "best_result.npz")
print("filtered MSPD fitness:", float(best["mspd_score"]))
print("raw MSPD:", float(best["raw_mspd_score"]))
print("DeltaH nonzero fraction:", float(best["delta_h_nonzero_frac"]))
print("passes DeltaH duration filter:", bool(best["passes_delta_h_filter"]))
print("trajectory:", best["trajectory"].shape, best["trajectory"].dtype)
print("DeltaH windows:", best["DeltaH"].shape[0])
print("scales:", best["mspd_scales"])
print("scale scores:", best["mspd_scale_scores"])

In [ ]:
from IPython.display import HTML, Image, display
from PIL import Image as PILImage

manifest_path = out_dir / "videos" / "video_manifest.csv"
if manifest_path.exists():
    display(pd.read_csv(manifest_path))

def trajectory_to_gif(label, trajectory, score=None, fps=12, scale=6, stride=2):
    gif_dir = out_dir / "videos" / "inline_gifs"
    gif_dir.mkdir(parents=True, exist_ok=True)
    gif_path = gif_dir / f"{label}.gif"
    frames = []
    for frame in trajectory[::stride]:
        img = (frame.astype("uint8") * 255)
        img = np.repeat(np.repeat(img, scale, axis=0), scale, axis=1)
        rgb = np.stack([img, img, img], axis=-1)
        frames.append(PILImage.fromarray(rgb))
    frames[0].save(
        gif_path,
        save_all=True,
        append_images=frames[1:],
        duration=max(1, int(1000 / fps)),
        loop=0,
    )
    suffix = "" if score is None else f" | MSPD={score:.6f}"
    display(HTML(f"<h4>{label}{suffix}</h4>"))
    display(Image(filename=str(gif_path)))
    return gif_path

best_result = result["best_result"]
trajectory_to_gif(
    "best_complex_life",
    best_result.trajectory,
    score=best_result.mspd_score,
    fps=CONFIG.video_fps,
    scale=max(4, CONFIG.video_scale // 2),
    stride=max(2, CONFIG.video_stride),
)

final_population = result["final_population"]
final_fitness = result["final_fitness"]
top_idx = np.argsort(final_fitness)[-min(CONFIG.video_top_k, len(final_population)):][::-1]
for rank, idx in enumerate(top_idx, start=1):
    trajectory_to_gif(
        f"final_population_complex_{rank:02d}",
        simulate_life(final_population[int(idx)], CONFIG.T),
        score=float(final_fitness[int(idx)]),
        fps=CONFIG.video_fps,
        scale=max(4, CONFIG.video_scale // 2),
        stride=max(2, CONFIG.video_stride),
    )

typical_idx = int(np.argmin(np.abs(final_fitness - np.median(final_fitness))))
trajectory_to_gif(
    "final_population_typical_life",
    simulate_life(final_population[typical_idx], CONFIG.T),
    score=float(final_fitness[typical_idx]),
    fps=CONFIG.video_fps,
    scale=max(4, CONFIG.video_scale // 2),
    stride=max(2, CONFIG.video_stride),
)

random_boards = result.get("random_boards")
random_scores = result.get("random_scores")
if random_boards is not None and len(random_boards):
    random_typical_idx = int(np.argmin(np.abs(random_scores - np.median(random_scores))))
    trajectory_to_gif(
        "random_typical_life",
        simulate_life(random_boards[random_typical_idx], CONFIG.T),
        score=float(random_scores[random_typical_idx]),
        fps=CONFIG.video_fps,
        scale=max(4, CONFIG.video_scale // 2),
        stride=max(2, CONFIG.video_stride),
    )

## ASAL-style Life-like rule sweep

This block searches over totalistic Life-like rules instead of initial boards. A rule is an 18-bit integer: bits `0..8` are birth outputs for a dead center with `0..8` live neighbors, and bits `9..17` are survival outputs for an alive center with `0..8` live neighbors. Conway Life is `6152 = B3/S23`.

In [ ]:
# This reload lets you run this new block without restarting the kernel.
import importlib
import gol_transition_mspd_experiment as gol_mspd
gol_mspd = importlib.reload(gol_mspd)
from gol_transition_mspd_experiment import lifelike_rule_label, run_rule_sweep_experiment

# All-rules ASAL-style enumeration: 262144 totalistic Life-like rules.
# This evaluates 262144 * N_RULE_INITIAL_BOARDS trajectories.
RUN_ALL_RULES = True
RULE_CANDIDATE_MODE = "all" if RUN_ALL_RULES else "random"  # "random", "linspace", or "all"
N_RULE_CANDIDATES = None if RUN_ALL_RULES else (4096 if RUN_PRESET == "large" else 512)
N_RULE_INITIAL_BOARDS = 6 if RUN_PRESET == "large" else 4  # same starts are used for every rule
RULE_INITIAL_DENSITY_RANGE = (0.05, 0.4)  # p ~ Uniform(low, high), then x_ij(0) ~ Bernoulli(p)

# Rule sweep cost controls. Increase RULE_EVAL_BATCH_SIZE only until memory stays stable.
RULE_T = CONFIG.T
RULE_BURN_IN = CONFIG.burn_in
RULE_WINDOW_SIZE = CONFIG.window_size
RULE_WINDOW_STEP = CONFIG.window_step
RULE_NULL_REPS = min(CONFIG.null_reps, 2)
RULE_PAIR_SAMPLE = 0 if CONFIG.pair_sample == 0 else min(CONFIG.pair_sample or 1024, 1024)
RULE_EVAL_BATCH_SIZE = 16 if RUN_PRESET == "large" else 64
RULE_JAX_METRIC_BATCH_SIZE = 0  # 0 = one vmap+jit metric call per rollout batch.
RULE_PROGRESS_INTERVAL = 64
INCLUDE_CONWAY_RULE = True

RULE_CONFIG = replace(
    CONFIG,
    T=RULE_T,
    burn_in=RULE_BURN_IN,
    window_size=RULE_WINDOW_SIZE,
    window_step=RULE_WINDOW_STEP,
    null_reps=RULE_NULL_REPS,
    pair_sample=RULE_PAIR_SAMPLE,
    eval_batch_size=RULE_EVAL_BATCH_SIZE,
    jax_metric_batch_size=RULE_JAX_METRIC_BATCH_SIZE,
    output_dir=str(REPO_ROOT / "analysis" / "results" / "gol_transition_mspd" / f"rule_sweep_{RUN_ID}"),
)

print("Conway:", 6152, lifelike_rule_label(6152))
RULE_CONFIG

Conway: 6152 B3/S23


ExperimentConfig(L=64, T=2048, burn_in=128, window_size=64, window_step=16, n_cell_sample=256, null_reps=2, population_size=64, generations=80, elite_frac=0.1, mutation_rate=0.01, initial_density=0.25, random_seed=0, backend='jax', eval_batch_size=512, jax_metric_batch_size=0, distance='js', pooled_null=True, pair_sample=1024, delta_h_floor=0.0, min_delta_h_nonzero_frac=0.5, delta_h_nonzero_eps=0.0, mspd_floor=1e-06, eps=1e-12, tournament_size=3, random_controls=64, output_dir='/Users/enrifermi/Projects/asal-fork/analysis/results/gol_transition_mspd/rule_sweep_20260522_031703', save_videos=True, video_top_k=3, video_fps=12, video_scale=8, video_stride=1, montage_frames=16, verbose=True, progress_every=1)

In [4]:
rule_result = run_rule_sweep_experiment(
    RULE_CONFIG,
    output_dir=RULE_CONFIG.output_dir,
    rule_candidate_mode=RULE_CANDIDATE_MODE,
    n_rule_candidates=N_RULE_CANDIDATES,
    n_initial_boards=N_RULE_INITIAL_BOARDS,
    initial_density_range=RULE_INITIAL_DENSITY_RANGE,
    include_conway=INCLUDE_CONWAY_RULE,
    stream_per_init_csv=True,
    progress_interval_rules=RULE_PROGRESS_INTERVAL,
    verbose=VERBOSE,
)
rule_result["summary"]

Starting ASAL-style Life-like rule sweep: rules=262144, n_initial_boards=6, L=64, T=2048, backend=jax, eval_batch_size=512, pair_sample=1024, min_delta_h_nonzero_frac>0.5, output_dir=/Users/enrifermi/Projects/asal-fork/analysis/results/gol_transition_mspd/rule_sweep_20260522_031703


: 

In [ ]:
import pandas as pd
from IPython.display import Image, display

rule_out_dir = Path(rule_result["summary"]["output_dir"])
rule_scores = pd.read_csv(rule_out_dir / "rule_sweep_scores.csv")
rule_per_init = pd.read_csv(rule_out_dir / "rule_sweep_per_init_scores.csv")

display(rule_scores.head(20))
display(rule_scores[rule_scores["rule_id"] == 6152])
display(rule_per_init.groupby(["rule_id", "rule_label"])["mspd_score"].describe().sort_values("mean", ascending=False).head(10))

for name in [
    "rule_sweep_top_scores.png",
    "rule_sweep_best_DeltaH_trace.png",
    "rule_sweep_best_life_montage.png",
]:
    display(Image(filename=str(rule_out_dir / "plots" / name)))

In [ ]:
if "trajectory_to_gif" not in globals():
    from IPython.display import HTML, Image, display
    from PIL import Image as PILImage
    def trajectory_to_gif(label, trajectory, score=None, fps=12, scale=6, stride=2):
        gif_dir = rule_out_dir / "inline_gifs"
        gif_dir.mkdir(parents=True, exist_ok=True)
        gif_path = gif_dir / f"{label}.gif"
        frames = []
        for frame in trajectory[::stride]:
            img = (frame.astype("uint8") * 255)
            img = np.repeat(np.repeat(img, scale, axis=0), scale, axis=1)
            frames.append(PILImage.fromarray(np.stack([img, img, img], axis=-1)))
        frames[0].save(gif_path, save_all=True, append_images=frames[1:], duration=max(1, int(1000 / fps)), loop=0)
        suffix = "" if score is None else f" | MSPD={score:.6f}"
        display(HTML(f"<h4>{label}{suffix}</h4>"))
        display(Image(filename=str(gif_path)))
        return gif_path

label = f"best_rule_{rule_result['best_rule_id']}_{rule_result['best_rule_label'].replace('/', '_')}"
trajectory_to_gif(
    label,
    rule_result["best_trajectory"],
    score=rule_result["summary"]["best_rule_mean_mspd_score"],
    fps=CONFIG.video_fps,
    scale=max(4, CONFIG.video_scale // 2),
    stride=max(2, CONFIG.video_stride),
)

In [ ]:
# Random representatives from an already completed rule sweep.
# This only re-simulates a small visual subset; it does not recompute MSPD.
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import HTML, Image, display
from PIL import Image as PILImage
from gol_transition_mspd_experiment import ExperimentConfig, simulate_lifelike_rule_batch

RANDOM_REPRESENTATIVES = 10
RANDOM_REP_SEED = 123
RANDOM_REP_REQUIRE_PASS = False  # set True to sample only rules with pass_fraction > 0
RANDOM_REP_BACKEND = "numpy"  # visual subset is small; numpy avoids another JAX compile

# If this is a fresh notebook after a terminal/GPU run, set this manually first:
# rule_out_dir = Path("analysis/results/gol_transition_mspd/cuda_all_rules")
if "rule_result" in globals():
    rule_out_dir = Path(rule_result["summary"]["output_dir"])
elif "rule_out_dir" not in globals():
    raise ValueError("Set rule_out_dir = Path('...your rule sweep output dir...') before running this cell.")

if "trajectory_to_gif" not in globals():
    def trajectory_to_gif(label, trajectory, score=None, fps=12, scale=6, stride=2):
        gif_dir = rule_out_dir / "inline_gifs"
        gif_dir.mkdir(parents=True, exist_ok=True)
        gif_path = gif_dir / f"{label}.gif"
        frames = []
        for frame in trajectory[::stride]:
            img = (frame.astype("uint8") * 255)
            img = np.repeat(np.repeat(img, scale, axis=0), scale, axis=1)
            frames.append(PILImage.fromarray(np.stack([img, img, img], axis=-1)))
        frames[0].save(gif_path, save_all=True, append_images=frames[1:], duration=max(1, int(1000 / fps)), loop=0)
        suffix = "" if score is None else f" | MSPD={score:.6f}"
        display(HTML(f"<h4>{label}{suffix}</h4>"))
        display(Image(filename=str(gif_path)))
        return gif_path

with (rule_out_dir / "rule_sweep_config.json").open() as f:
    rule_config_json = json.load(f)
VIS_RULE_CONFIG = RULE_CONFIG if "RULE_CONFIG" in globals() else ExperimentConfig(**rule_config_json["base_config"])
VIDEO_CONFIG = CONFIG if "CONFIG" in globals() else VIS_RULE_CONFIG
RANDOM_REP_STEPS = min(int(VIS_RULE_CONFIG.T), 1024)
RANDOM_REP_STRIDE = max(4, int(np.ceil(RANDOM_REP_STEPS / 256)))

rule_scores = pd.read_csv(rule_out_dir / "rule_sweep_scores.csv")
best_npz = np.load(rule_out_dir / "best_rule_result.npz", allow_pickle=True)
initial_boards_for_rules = best_npz["initial_boards"].astype(np.uint8)

sample_pool = rule_scores.copy()
if RANDOM_REP_REQUIRE_PASS and "pass_fraction" in sample_pool.columns:
    sample_pool = sample_pool[sample_pool["pass_fraction"] > 0].copy()
if len(sample_pool) == 0:
    raise ValueError("No rules available for random representative sampling.")

rng = np.random.default_rng(RANDOM_REP_SEED)
n_random = min(RANDOM_REPRESENTATIVES, len(sample_pool))
selected = sample_pool.sample(n=n_random, replace=False, random_state=RANDOM_REP_SEED).reset_index(drop=True)
selected["init_id"] = rng.integers(0, initial_boards_for_rules.shape[0], size=n_random)

boards = np.stack([initial_boards_for_rules[int(init_id)] for init_id in selected["init_id"]], axis=0)
rules = selected["rule_id"].to_numpy(dtype=np.uint32)
trajectories = simulate_lifelike_rule_batch(boards, rules, RANDOM_REP_STEPS, backend=RANDOM_REP_BACKEND)

display(selected[["rule_id", "rule_label", "init_id", "mean_mspd", "max_mspd", "pass_fraction"]])
display(HTML(f"<p>Rendering {n_random} random representatives: steps={RANDOM_REP_STEPS}, stride={RANDOM_REP_STRIDE}</p>"))

for idx, row in selected.iterrows():
    safe_label = str(row["rule_label"]).replace("/", "_")
    label = f"random_rule_{idx:02d}_{int(row['rule_id'])}_{safe_label}_init{int(row['init_id'])}"
    trajectory_to_gif(
        label,
        trajectories[idx],
        score=float(row["mean_mspd"]),
        fps=VIDEO_CONFIG.video_fps,
        scale=max(4, VIDEO_CONFIG.video_scale // 2),
        stride=RANDOM_REP_STRIDE,
    )